In [ ]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.cloud import storage

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, StructType, StructField, IntegerType, StringType, DateType

from utils.PostParser import post_parser
from utils.LocationFunctions import load_locations_df, get_locations_from_bq, get_missing_locations, get_batch_geocode, update_locations_bq
from utils.Common import gcs_file_read, gcs_upload_parquet, partial_parse_raw_data

gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

load_dotenv()

RawSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

In [ ]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Transform Stage (Kaggle)') \
        .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
        .getOrCreate() 

project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
locations_table_id = f"{project_id}:{dataset}.dim_locations"
staging_locations_table_id = f"{project_id}:{dataset}.staging_locations"
bucket_name = os.getenv('BUCKET_NAME')
raw_folder = os.getenv('RAW_FOLDER_NAME')
clean_folder = os.getenv('CLEANED_FOLDER_NAME')

kaggle_folder=f"{raw_folder}/kaggle"

gcs_client = storage.Client()
bucket = gcs_client.bucket(bucket_name)

df_locations = load_locations_df(spark, locations_table_id)
raw_filename = f"{kaggle_folder}/kaggle_historical_data.csv"
is_file_exists = storage.Blob(bucket=bucket, name=raw_filename).exists()

if not is_file_exists:
    print(f"{raw_filename} does not exists")

df_raw = gcs_file_read(spark, bucket_name, raw_filename, RawSchema)
df_raw_specific = df_raw.select("Tweet", "Date", "Source")
df_raw_renamed = df_raw_specific.withColumnsRenamed({"Tweet": "content", "Date": "created_at", "Source": "tweetlinkid"})

In [ ]:
df_partial_parsed = partial_parse_raw_data(df_raw_renamed)

In [ ]:
df_partial_parsed.printSchema()

In [ ]:
df_partial_parsed.show(60, truncate=False)

In [ ]:
df_partial_parsed.filter(F.col("timestamp").isNull()).count()

In [ ]:
df_partial_parsed.filter(F.col("timestamp").isNull()).show(10, truncate=False)

In [ ]:
df_filled = df_partial_parsed.withColumn(
    "event_timestamp", 
    F.to_timestamp(F.concat_ws(' ', F.col("date"), F.col("time")), "yyyy-MM-dd HH:mm"))

In [ ]:
df_filled.show(5, truncate=False)

In [ ]:
from pyspark.sql.window import Window

window = Window.rowsBetween(0, Window.unboundedFollowing)

In [ ]:
window2 = Window.partitionBy("date").orderBy(F.col("timestamp").asc()).rowsBetween(0, Window.unboundedFollowing)

In [ ]:
df_filled = df_partial_parsed.withColumns({
    "filled_time": F.first("time", ignorenulls=True).over(window),
    "filled_hour": F.first("hour", ignorenulls=True).over(window),
})

In [ ]:
df_filled.show(60, truncate=False)

In [ ]:
df_filled_2 = df_partial_parsed.withColumn("filled_time", F.first("time", ignorenulls=True).over(window2))

In [ ]:
df_filled_2.show(60, truncate=False)

In [ ]:
window3 = Window.partitionBy("date").orderBy(F.col("date").asc()).rowsBetween(0, Window.unboundedFollowing)

df_filled_3 = df_partial_parsed.withColumn("filled_time", F.first("time", ignorenulls=True).over(window3))

In [ ]:
df_filled_3.show(60, truncate=False)

In [ ]:
window4 = Window.rowsBetween(0, Window.unboundedFollowing)

df_filled_4 = df_partial_parsed.withColumn("filled_time", F.first("time", ignorenulls=True).over(window4))

In [ ]:
df_filled_4.show(60, truncate=False)

In [ ]:
window5 = Window.rowsBetween(Window.unboundedPreceding, 0)

df_filled_5 = df_partial_parsed.withColumn("filled_time", F.last("time", ignorenulls=True).over(window5))

In [ ]:
df_filled_5.show(60, truncate=False)

In [1]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.cloud import storage

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, StructType, StructField, IntegerType, StringType, DateType

from utils.LocationFunctions import load_locations_df, get_locations_from_bq, get_missing_locations, get_batch_geocode, update_locations_bq
from utils.Common import gcs_file_read, gcs_upload_parquet, partial_parse_raw_data

gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

load_dotenv()

KaggleRawSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

def initialize(app_name, data_source='scraped'):
    """Initialize Spark, GCS client, and location references for the transform pipeline.

    Args:
        app_name: Name for the Spark application.
        data_source: Data source type ('scraped' or 'kaggle').

    Returns:
        tuple: spark, gcs_client, project_id, dataset, locations_table_id,
               staging_locations_table_id, bucket_name, raw_folder,
               clean_folder, scrape_folder, df_locations
    """
    spark = SparkSession.builder \
            .master("local[*]") \
            .appName(app_name) \
            .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
            .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
            .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
            .getOrCreate()
    
    gcs_client = storage.Client()

    project_id = os.getenv("PROJECT_ID")
    dataset = os.getenv("DATASET")
    locations_table_id = f"{project_id}:{dataset}.dim_locations"
    staging_locations_table_id = f"{project_id}:{dataset}.staging_locations"
    bucket_name = os.getenv('BUCKET_NAME')
    raw_folder = os.getenv('RAW_FOLDER_NAME')
    clean_folder = os.getenv('CLEANED_FOLDER_NAME')
    scrape_folder = f"{raw_folder}/scrape"

    df_locations = load_locations_df(spark, locations_table_id)

    return spark, gcs_client, project_id, dataset, locations_table_id, staging_locations_table_id, bucket_name, raw_folder, clean_folder, scrape_folder, df_locations


def process_df(spark, df_raw, df_locations, locations_table_id, staging_locations_table_id, project_id, dataset):
    """
    Process a DataFrame through the transformation pipeline
    
    Args:
        spark: SparkSession
        df_raw: Raw DataFrame
        df_locations: Locations DataFrame
        locations_table_id: BigQuery locations table ID
        staging_locations_table_id: BigQuery staging locations table ID
        project_id: GCP project ID
        dataset: BigQuery dataset name
        
    Returns:
        Processed DataFrame
    """
    # parse
    df_partial_parsed = partial_parse_raw_data(df_raw)

    # handle missing timestamps
    null_timestamp_count = df_partial_parsed.filter(F.col("timestamp").isNull()).count()
    if null_timestamp_count > 0:
        window = Window.rowsBetween(Window.unboundedPreceding, 0)
        df_partial_parsed = df_partial_parsed.withColumn("time", F.last("time", ignorenulls=True).over(window))
        df_partial_parsed = df_partial_parsed.withColumn(
            "timestamp", 
            F.to_timestamp(F.concat_ws(' ', F.col("date"), F.col("time")), "yyyy-MM-dd HH:mm")
        )

    # enrich from BQ
    df_full_parsed = get_locations_from_bq(df_locations, df_partial_parsed)

    print("df_full_parsed after BQ enrichment:")
    df_full_parsed.show(5, truncate=False)

    # handle missing locations
    missing_locations = get_missing_locations(df_full_parsed)
    print(f"Missing locations count: {len(missing_locations)}")

    if len(missing_locations) != 0:
        resolved_locations_df = get_batch_geocode(spark, missing_locations)
        print(f"Resolved locations count: {resolved_locations_df.count()}")

        print("Resolved locations from geocoding API:")
        resolved_locations_df.show(5, truncate=False)

        update_locations_bq(resolved_locations_df, staging_locations_table_id, project_id, dataset)

        # reload updated locations
        df_locations = load_locations_df(spark, locations_table_id)

        df_full_parsed = df_full_parsed.drop("city", "latitude", "longitude", "accuracy")
        df_full_parsed = get_locations_from_bq(df_locations, df_full_parsed)

    df_final = df_full_parsed.select(
        'date', 'time', 'timestamp', 'location_id',
        'city', 'location', 'latitude', 'longitude', 'accuracy',
        'direction', 'type', 'lanes_blocked',
        'involved', 'post', 'link'
    )

    return df_final


def process_kaggle_file(spark, gcs_client, bucket_name, raw_filename, df_locations, locations_table_id, staging_locations_table_id, project_id, dataset):
    """
    Process the Kaggle raw data file through the transformation pipeline
    
    Args:
        spark: SparkSession
        bucket_name: GCS bucket name
        raw_filename: GCS path to raw file
        df_locations: Locations DataFrame
        locations_table_id: BigQuery locations table ID
        staging_locations_table_id: BigQuery staging locations table ID
        project_id: GCP project ID
        dataset: BigQuery dataset name
        
    Returns:
        Processed DataFrame or None if file doesn't exist
    """
    bucket = gcs_client.bucket(bucket_name)
    
    blob = storage.Blob(bucket=bucket, name=raw_filename)
    if not blob.exists():
        print(f"Skipping (not found): {raw_filename}")
        return None

    print(f"Processing: {raw_filename}")
    df_raw = gcs_file_read(spark, bucket_name, raw_filename, KaggleRawSchema)
    df_raw_specific = df_raw.select("Tweet", "Date", "Source")
    df_raw_renamed = df_raw_specific.withColumnsRenamed({"Tweet": "content", "Date": "created_at", "Source": "tweetlinkid"})

    df_raw_renamed.show(5, truncate=False)

    df_final = process_df(spark, df_raw_renamed, df_locations, locations_table_id, staging_locations_table_id, project_id, dataset)

    return df_final


def run_kaggle_transform():
    """Run transformation for Kaggle historical data.

    This function initializes Spark and GCS, processes the Kaggle raw file,
    transforms it, and uploads the cleaned parquet output to GCS.
    """
    spark, gcs_client, project_id, dataset, locations_table_id, staging_locations_table_id, bucket_name, raw_folder, clean_folder, scrape_folder, df_locations = initialize('Transform Stage (Kaggle)', 'kaggle')

    kaggle_folder = f"{raw_folder}/kaggle"
    raw_filename = f"{kaggle_folder}/kaggle_historical_data.csv"
    
    df_final = process_kaggle_file(
        spark, gcs_client, bucket_name, raw_filename, df_locations,
        locations_table_id, staging_locations_table_id, project_id, dataset
    )
    
    if df_final is None:
        print(f"{raw_filename} does not exist")
        return

    # upload to gcs as parquet
    # gcs_upload_parquet(bucket_name, clean_folder, df_final)

    print("Kaggle processing complete.")
    return df_final



In [2]:
df_kaggle = run_kaggle_transform()

26/05/02 15:43:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Processing: raw/kaggle/kaggle_historical_data.csv


+---------------------------------------------------------------------------------------------------------------------------------+----------+---------------------------------------------------+
|content                                                                                                                          |created_at|tweetlinkid                                        |
+---------------------------------------------------------------------------------------------------------------------------------+----------+---------------------------------------------------+
|MMDA ALERT: Vehicular accident at Ortigas Emerald EB  involving taxi and MC as of 7:55 AM. 1 lane occupied. MMDA on site. #mmda  |2018-08-20|https://twitter.com/mmda/status/1031330201970532352|
|MMDA ALERT: Stalled L300 due to mechanical problem at EDSA Guadix NB as of 8:42 AM. 1 lane occupied. MMDA enforcer on site. #mmda|2018-08-20|https://twitter.com/mmda/status/1031346247745990656|
|MMDA ALERT: Vehicular ac

df_full_parsed after BQ enrichment:


26/05/02 15:43:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:43:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:43:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:43:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:43:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:43:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------------+----------------------------------------------------------------+-----------+---------+----------+--------+----------+-----+-------------------+---------+--------------------------------------+-------------+------------+---------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------+
|location             |location_id                                                     |city       |latitude |longitude |accuracy|date      |time |timestamp          |direction|type                                  |lanes_blocked|involved    |post                                                                                                                             |link                                               |
+---------------------+----------------------------------------------------------------+-----------+---------+----------+--------+----------+-----+-

Missing locations count: 0
Kaggle processing complete.


In [3]:
df_kaggle.show(5, truncate=False)

26/05/02 15:44:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:44:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:44:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:44:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----+-------------------+----------------------------------------------------------------+-----------+---------------------+---------+----------+--------+---------+--------------------------------------+-------------+------------+---------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------+
|date      |time |timestamp          |location_id                                                     |city       |location             |latitude |longitude |accuracy|direction|type                                  |lanes_blocked|involved    |post                                                                                                                             |link                                               |
+----------+-----+-------------------+----------------------------------------------------------------+-----------+---------------------+---------+-

26/05/02 15:44:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 15:44:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [4]:
df_kaggle.filter(F.col("location_id").isNull()).count()

0